# 🧠 ENGRAMA 30M - Dual T4 GPU Training on TinyStories
### Complete Pure PyTorch Architecture Implementation for Kaggle Dual GPU Setup

- **Architecture:** ENGRAMA (Phase 1 IsolatedEncoder, Phase 2 Trace Cache, Phase 3 Dilated Consolidation Stack, Phase 4 Multi-Candidate Evoker)
- **Model Size:** ~32.3 Million Parameters (`d_model=384`, `d_gate=48`, `d_ff=1536`, `num_cells=4`, `num_encoder_layers=1`, `num_consolidation_layers=2`)
- **Dataset:** `roneneldan/TinyStories` (Complete Dataset)
- **Tokenizer:** GPT-2 Tokenizer (`vocab_size=50257`)
- **Context Length:** 512 tokens
- **Hardware:** Dual NVIDIA Tesla T4 GPUs (Kaggle Multi-GPU setup)
- **Training Epochs:** 1 Full Epoch
- **No Attention / $QK^T$:** Pure cellular, dynamic gating, and logarithmic dilated consolidation architecture.


In [ ]:
# Step 1: Environment Verification and Package Installation
import os
import sys
import math
import time
import json
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Dict, Any, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Install required Hugging Face utilities for dataset streaming & tokenization
!pip install -q datasets transformers accelerate tqdm

import datasets
from transformers import AutoTokenizer
from tqdm.auto import tqdm

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# Step 2: ENGRAMA Pure PyTorch Core Architecture Definition
# Implements exact mathematical specifications from ENGRAMA paper (No HF architecture dependencies)

@dataclass
class EngramaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    d_gate: int = 48
    d_ff: int = 1536
    num_cells: int = 4
    num_encoder_layers: int = 1
    num_consolidation_layers: int = 2
    context_length: int = 512
    offsets: Optional[List[int]] = None
    num_candidates: int = 4
    candidate_aggregation: str = "logsumexp"
    activation: str = "gelu"
    dropout: float = 0.0
    dtype: str = "float32"
    version: str = "v2"
    tie_embeddings: bool = True

    def __post_init__(self):
        if self.offsets is None:
            self.offsets = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]
        if self.d_ff is None:
            self.d_ff = 4 * self.d_model

class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

class Cell(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0, activation: str = "gelu"):
        super().__init__()
        self.ln = LayerNorm(d_model)
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU() if activation == "gelu" else (nn.SiLU() if activation == "silu" else nn.ReLU())
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = x
        x_norm = self.ln(x)
        h = self.act(self.w1(x_norm))
        h = self.dropout(h)
        out = self.w2(h)
        return res + out

class SynapseLayer(nn.Module):
    def __init__(self, d_model: int, d_gate: int, num_cells: int = 4, d_ff: int = 1536, dropout: float = 0.0, activation: str = "gelu"):
        super().__init__()
        self.C = num_cells
        self.d_model = d_model
        self.d_gate = d_gate
        self.cells = nn.ModuleList([Cell(d_model, d_ff, dropout, activation) for _ in range(num_cells)])
        self.gate_w = nn.Parameter(torch.randn(num_cells, num_cells, d_gate) * 0.02)
        self.gate_b = nn.Parameter(torch.zeros(num_cells, num_cells))
        self.w_channel = nn.Parameter(torch.randn(num_cells, num_cells, d_model) * 0.02)
        self.p_g = nn.Linear(d_model, d_gate, bias=False)
        self.w_transform = nn.Linear(d_model, d_model, bias=False)

    def forward(self, cell_outputs: List[torch.Tensor]) -> List[torch.Tensor]:
        H_stack = torch.stack(cell_outputs, dim=-2)
        g_proj = self.p_g(H_stack)
        next_outputs = []
        for j in range(self.C):
            H_transformed = self.w_transform(cell_outputs[j])
            channel_contributions = []
            for i in range(self.C):
                g_i = g_proj[..., i, :]
                w_ij = self.gate_w[i, j]
                b_ij = self.gate_b[i, j]
                alpha_ij = torch.sigmoid(torch.einsum("...g,g->...", g_i, w_ij) + b_ij)
                w_channel_ij = self.w_channel[i, j]
                h_i = cell_outputs[i]
                contrib = alpha_ij.unsqueeze(-1) * (h_i * w_channel_ij)
                channel_contributions.append(contrib)
            synapse_mix = torch.stack(channel_contributions, dim=0).sum(dim=0)
            inp_j = H_transformed + synapse_mix
            out_j = self.cells[j](inp_j)
            next_outputs.append(out_j)
        return next_outputs

class IsolatedEncoder(nn.Module):
    def __init__(self, d_model: int, d_gate: int, num_cells: int, num_encoder_layers: int, d_ff: int, dropout: float, activation: str):
        super().__init__()
        self.init_proj = nn.Linear(d_model, num_cells * d_model)
        self.num_cells = num_cells
        self.d_model = d_model
        self.layers = nn.ModuleList([
            SynapseLayer(d_model, d_gate, num_cells, d_ff, dropout, activation)
            for _ in range(num_encoder_layers)
        ])
        self.w_pool = nn.Linear(num_cells * d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        proj = self.init_proj(x)
        cells = [proj[..., i * D : (i + 1) * D] for i in range(self.num_cells)]
        for layer in self.layers:
            cells = layer(cells)
        pooled = torch.cat(cells, dim=-1)
        T0 = self.w_pool(pooled)
        return T0

class PositionalDilatedMix(nn.Module):
    def __init__(self, d_model: int, d_gate: int, offsets: List[int]):
        super().__init__()
        self.d_model = d_model
        self.d_gate = d_gate
        self.offsets = list(offsets)
        self.p_g = nn.Linear(d_model, d_gate, bias=False)
        self.w_offsets = nn.ModuleDict({str(p): nn.Linear(d_model, d_model, bias=False) for p in self.offsets})
        self.gate_w = nn.ParameterDict({str(p): nn.Parameter(torch.randn(d_gate, d_model) * 0.01) for p in self.offsets})
        self.gate_b = nn.ParameterDict({str(p): nn.Parameter(torch.zeros(d_model)) for p in self.offsets})

    def forward_train(self, T_prev: torch.Tensor) -> torch.Tensor:
        b, n, d = T_prev.shape
        t_pos = torch.zeros_like(T_prev)
        for p in self.offsets:
            str_p = str(p)
            if p == 0:
                t_shifted = T_prev
            elif p < n:
                t_shifted = torch.cat([
                    torch.zeros(b, p, d, device=T_prev.device, dtype=T_prev.dtype),
                    T_prev[:, :-p, :]
                ], dim=1)
            else:
                continue
            h_g = self.p_g(t_shifted)
            gate_logits = torch.matmul(h_g, self.gate_w[str_p]) + self.gate_b[str_p]
            g = torch.sigmoid(gate_logits)
            transformed = self.w_offsets[str_p](t_shifted)
            t_pos = t_pos + g * transformed
        return t_pos

    def forward_step(self, T_prev_history: Union[torch.Tensor, List[torch.Tensor]], current_pos: int) -> torch.Tensor:
        if isinstance(T_prev_history, torch.Tensor):
            b, n_hist, d = T_prev_history.shape
            device, dtype = T_prev_history.device, T_prev_history.dtype
        else:
            first = T_prev_history[0]
            b, d = first.shape[0], first.shape[-1]
            device, dtype = first.device, first.dtype

        t_pos = torch.zeros(b, d, device=device, dtype=dtype)
        for p in self.offsets:
            idx = current_pos - p
            if idx < 0:
                continue
            if isinstance(T_prev_history, torch.Tensor):
                if idx >= T_prev_history.shape[1]: continue
                t_p = T_prev_history[:, idx, :]
            else:
                if idx >= len(T_prev_history): continue
                t_p = T_prev_history[idx]
                if t_p.dim() == 3: t_p = t_p.squeeze(1)

            str_p = str(p)
            h_g = self.p_g(t_p)
            gate_logits = torch.matmul(h_g, self.gate_w[str_p]) + self.gate_b[str_p]
            g = torch.sigmoid(gate_logits)
            transformed = self.w_offsets[str_p](t_p)
            t_pos = t_pos + g * transformed
        return t_pos

class ConsolidationLayer(nn.Module):
    def __init__(self, d_model: int, d_gate: int, offsets: List[int], d_ff: int, dropout: float = 0.0, activation: str = "gelu"):
        super().__init__()
        self.mix = PositionalDilatedMix(d_model, d_gate, offsets)
        self.cell = Cell(d_model, d_ff, dropout, activation)

    def forward_train(self, T_prev: torch.Tensor) -> torch.Tensor:
        t_pos = self.mix.forward_train(T_prev)
        return self.cell(t_pos)

    def forward_step(self, T_prev_history: Union[torch.Tensor, List[torch.Tensor]], current_pos: int) -> torch.Tensor:
        t_pos = self.mix.forward_step(T_prev_history, current_pos)
        return self.cell(t_pos)

class ConsolidationStack(nn.Module):
    def __init__(self, d_model: int, d_gate: int, offsets: List[int], num_consolidation_layers: int, d_ff: int, dropout: float = 0.0, activation: str = "gelu"):
        super().__init__()
        self.layers = nn.ModuleList([
            ConsolidationLayer(d_model, d_gate, offsets, d_ff, dropout, activation)
            for _ in range(num_consolidation_layers)
        ])

    def forward_train(self, T0: torch.Tensor) -> torch.Tensor:
        t = T0
        for layer in self.layers:
            t = layer.forward_train(t)
        return t

    def step_forward(self, cache: Any, T0_current: Optional[torch.Tensor] = None, return_all_layers: bool = False):
        if T0_current is not None:
            history_t0 = cache.T0 + [T0_current]
            current_pos = len(cache.T0)
        else:
            history_t0 = cache.T0
            current_pos = len(cache.T0) - 1

        layer_outputs = []
        for l, layer in enumerate(self.layers):
            prev_hist = history_t0 if l == 0 else cache.Tl[l - 1] + [layer_outputs[l - 1]]
            t_out = layer.forward_step(prev_hist, current_pos)
            layer_outputs.append(t_out)

        t_l = layer_outputs[-1]
        return (t_l, layer_outputs) if return_all_layers else t_l

class MultiCandidateEvoker(nn.Module):
    def __init__(self, d_model: int, vocab_size: int, num_candidates: int = 4, aggregation: str = "logsumexp"):
        super().__init__()
        self.num_candidates = num_candidates
        self.aggregation = aggregation
        self.candidates = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(num_candidates)])

    def forward(self, T_L: torch.Tensor, embedding_weight: torch.Tensor) -> torch.Tensor:
        candidate_logits = []
        for cand in self.candidates:
            h_c = cand(T_L)
            logits_c = F.linear(h_c, embedding_weight)
            candidate_logits.append(logits_c)
        stacked = torch.stack(candidate_logits, dim=-1)
        if self.aggregation == "logsumexp":
            return torch.logsumexp(stacked, dim=-1)
        elif self.aggregation == "max":
            return torch.max(stacked, dim=-1)[0]
        else:
            return torch.mean(stacked, dim=-1)

class EngramaCache:
    def __init__(self, N_max: int, num_layers: int, d_model: int):
        self.N_max = N_max
        self.num_layers = num_layers
        self.d_model = d_model
        self.timestamps = []
        self.T0 = []
        self.Tl = [[] for _ in range(num_layers)]

    def append(self, T0_t: torch.Tensor, Tl_t_layers: List[torch.Tensor], timestamp: int):
        if len(self.timestamps) >= self.N_max:
            self.timestamps.pop(0)
            self.T0.pop(0)
            for l in range(self.num_layers):
                self.Tl[l].pop(0)
        self.timestamps.append(timestamp)
        self.T0.append(T0_t)
        for l in range(self.num_layers):
            self.Tl[l].append(Tl_t_layers[l])

    def __len__(self):
        return len(self.timestamps)

class EngramaModel(nn.Module):
    def __init__(self, config: EngramaConfig):
        super().__init__()
        self.config = config
        self.embeddings = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = IsolatedEncoder(
            d_model=config.d_model,
            d_gate=config.d_gate,
            num_cells=config.num_cells,
            num_encoder_layers=config.num_encoder_layers,
            d_ff=config.d_ff,
            dropout=config.dropout,
            activation=config.activation,
        )
        self.consolidation = ConsolidationStack(
            d_model=config.d_model,
            d_gate=config.d_gate,
            offsets=config.offsets,
            num_consolidation_layers=config.num_consolidation_layers,
            d_ff=config.d_ff,
            dropout=config.dropout,
            activation=config.activation,
        )
        self.evoker = MultiCandidateEvoker(
            d_model=config.d_model,
            vocab_size=config.vocab_size,
            num_candidates=config.num_candidates,
            aggregation=config.candidate_aggregation,
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embeddings(input_ids)
        T0 = self.encoder(x)
        T_L = self.consolidation.forward_train(T0)
        logits = self.evoker(T_L, self.embeddings.weight)
        return logits

    def step_forward(self, token_id: torch.Tensor, cache: EngramaCache, timestamp: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if token_id.dim() == 1:
            token_id = token_id.unsqueeze(1)
        elif token_id.dim() == 0:
            token_id = token_id.unsqueeze(0).unsqueeze(0)

        x_t = self.embeddings(token_id)
        t0_t = self.encoder(x_t)
        if t0_t.dim() == 3:
            t0_t = t0_t.squeeze(1)

        t_l_t, layer_outputs = self.consolidation.step_forward(cache, T0_current=t0_t, return_all_layers=True)
        logits_t = self.evoker(t_l_t, self.embeddings.weight)
        cache.append(t0_t, layer_outputs, timestamp)
        return logits_t, t_l_t

    def num_parameters(self, only_trainable: bool = False) -> int:
        if only_trainable:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("ENGRAMA Architecture loaded successfully!")


In [ ]:
# Step 3: Instantiate ~32.3M Parameter ENGRAMA Model
config = EngramaConfig(
    vocab_size=50257,               # GPT-2 Tokenizer Vocab Size
    d_model=384,                    # Hidden Representation Dimension
    d_gate=48,                      # Gating Latent Projection Dimension
    d_ff=1536,                      # Cell Feed-Forward Expansion Dimension (4 * d_model)
    num_cells=4,                    # Number of Cellular Representations in Synapse Layer
    num_encoder_layers=1,           # IsolatedEncoder Synapse Layers
    num_consolidation_layers=2,     # Consolidation Stack Layers
    context_length=512,             # Maximum Context Sequence Length
    offsets=[0, 1, 2, 4, 8, 16, 32, 64, 128, 256], # Logarithmic Dilated Offsets
    num_candidates=4,               # Recall Candidates in MultiCandidateEvoker
    candidate_aggregation="logsumexp",
    activation="gelu",
    dropout=0.1,
    version="v2",
    tie_embeddings=True
)

model = EngramaModel(config)
total_params = model.num_parameters()

print("=" * 60)
print(f"  ENGRAMA 30M MODEL PARAMETER BREAKDOWN")
print("=" * 60)
print(f"  Vocab Size              : {config.vocab_size}")
print(f"  Embedding Dimension (d) : {config.d_model}")
print(f"  Gate Dimension (d_gate) : {config.d_gate}")
print(f"  Feed-Forward Dimension  : {config.d_ff}")
print(f"  Number of Cells (C)     : {config.num_cells}")
print(f"  Encoder Synapse Layers  : {config.num_encoder_layers}")
print(f"  Consolidation Layers    : {config.num_consolidation_layers}")
print(f"  Context Sequence Length : {config.context_length}")
print(f"  Total Parameters        : {total_params:,} ({total_params/1e6:.2f} Million)")
print("=" * 60)


In [ ]:
# Step 4: Load TinyStories Dataset & Tokenize with GPT-2 Tokenizer
print("Loading GPT-2 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Loading roneneldan/TinyStories dataset from Hugging Face...")
raw_dataset = datasets.load_dataset("roneneldan/TinyStories")

print(f"Train split size: {len(raw_dataset['train']):,} stories")
print(f"Validation split size: {len(raw_dataset['validation']):,} stories")

SEQ_LEN = 512

class TinyStoriesTokenDataset(Dataset):
    def __init__(self, raw_data, tokenizer, seq_len=512, max_samples=None):
        self.seq_len = seq_len
        texts = raw_data['text']
        if max_samples:
            texts = texts[:max_samples]
        
        print("Tokenizing stories into contiguous tokens...")
        full_tokens = []
        for text in tqdm(texts, desc="Tokenizing"):
            tokens = tokenizer.encode(text, add_special_tokens=True)
            full_tokens.extend(tokens)
            full_tokens.append(tokenizer.eos_token_id)
            
        total_tokens = len(full_tokens)
        num_chunks = total_tokens // (seq_len + 1)
        truncated_len = num_chunks * (seq_len + 1)
        
        self.data = torch.tensor(full_tokens[:truncated_len], dtype=torch.long).view(-1, seq_len + 1)
        print(f"Created {len(self.data):,} sequences of length {seq_len} ({len(self.data) * seq_len:,} total tokens)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk = self.data[idx]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

print("Preparing Train Dataset...")
train_dataset = TinyStoriesTokenDataset(raw_dataset['train'], tokenizer, seq_len=SEQ_LEN)

print("Preparing Validation Dataset...")
val_dataset = TinyStoriesTokenDataset(raw_dataset['validation'], tokenizer, seq_len=SEQ_LEN)


In [ ]:
# Step 5: Multi-GPU Dual NVIDIA Tesla T4 Engine Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"🚀 Harnessing Dual Tesla T4 GPUs! Using torch.nn.DataParallel across {torch.cuda.device_count()} GPUs.")
    train_model = nn.DataParallel(model)
else:
    print(f"Running on single GPU / CPU: {device}")
    train_model = model

train_model = train_model.to(device)

BATCH_SIZE_PER_GPU = 16
NUM_GPUS = max(1, torch.cuda.device_count())
TOTAL_BATCH_SIZE = BATCH_SIZE_PER_GPU * NUM_GPUS

print(f"Batch Size per GPU : {BATCH_SIZE_PER_GPU}")
print(f"Total Effective Batch Size : {TOTAL_BATCH_SIZE}")

train_loader = DataLoader(
    train_dataset,
    batch_size=TOTAL_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TOTAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    drop_last=False
)


In [ ]:
# Step 6: Optimizer, Learning Rate Scheduler & Training Engine Setup
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
EPOCHS = 1

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95)
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.05 * total_steps)

def get_lr_factor(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_factor)
scaler = torch.cuda.amp.GradScaler()
criterion = nn.CrossEntropyLoss()

print(f"Total Training Steps : {total_steps:,}")
print(f"Warmup Steps         : {warmup_steps:,}")


In [ ]:
# Step 7: Train ENGRAMA 30M Model for 1 Full Epoch on Dual T4 GPUs
print("\n" + "=" * 70)
print("  STARTING ENGRAMA 30M DUAL T4 GPU TRAINING (1 EPOCH)")
print("=" * 70)

train_model.train()
start_time = time.time()
running_loss = 0.0
tokens_processed = 0

pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc="Epoch 1/1 Training")

for step, (input_ids, labels) in pbar:
    input_ids = input_ids.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    
    optimizer.zero_grad()
    
    # Mixed precision forward pass
    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits = train_model(input_ids)
        loss = criterion(logits.view(-1, config.vocab_size), labels.view(-1))
    
    # Mixed precision backward pass & gradient clipping
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(train_model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    
    step_loss = loss.item()
    running_loss += step_loss
    batch_tokens = input_ids.numel()
    tokens_processed += batch_tokens
    
    current_lr = optimizer.param_groups[0]['lr']
    
    if (step + 1) % 100 == 0 or step == len(train_loader) - 1:
        elapsed = time.time() - start_time
        tok_per_sec = tokens_processed / elapsed
        avg_loss = running_loss / (step + 1)
        perplexity = math.exp(min(avg_loss, 20.0))
        
        pbar.set_postfix({
            "Loss": f"{avg_loss:.4f}",
            "PPL": f"{perplexity:.2f}",
            "LR": f"{current_lr:.2e}",
            "Tok/s": f"{tok_per_sec:.0f}"
        })

total_elapsed = time.time() - start_time
final_avg_loss = running_loss / len(train_loader)
final_ppl = math.exp(min(final_avg_loss, 20.0))

print("\n" + "=" * 70)
print(f"  EPOCH 1 TRAINING COMPLETED IN {total_elapsed/60:.2f} MINUTES")
print(f"  Final Training Loss : {final_avg_loss:.4f}")
print(f"  Final Perplexity    : {final_ppl:.2f}")
print(f"  Total Processed     : {tokens_processed:,} tokens ({tokens_processed/total_elapsed:.0f} tok/sec)")
print("=" * 70)


In [ ]:
# Step 8: Validation Split Evaluation
train_model.eval()
val_loss = 0.0
val_tokens = 0

print("\nEvaluating model on Validation split...")
with torch.no_grad():
    for input_ids, labels in tqdm(val_loader, desc="Validation"):
        input_ids = input_ids.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            logits = train_model(input_ids)
            loss = criterion(logits.view(-1, config.vocab_size), labels.view(-1))
            
        val_loss += loss.item() * input_ids.size(0)
        val_tokens += input_ids.size(0)

avg_val_loss = val_loss / val_tokens
val_ppl = math.exp(min(avg_val_loss, 20.0))

print("=" * 60)
print(f"  VALIDATION RESULTS")
print("=" * 60)
print(f"  Validation Loss       : {avg_val_loss:.4f}")
print(f"  Validation Perplexity : {val_ppl:.2f}")
print("=" * 60)


In [ ]:
# Step 9: Autoregressive Text Generation Demo & Causal Invariance Test
model.eval()
model.to(device)

def generate_story(prompt: str, max_new_tokens: int = 100, temperature: float = 0.8, top_k: int = 40):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids[0].tolist()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            inp_tensor = torch.tensor([generated[-config.context_length:]], dtype=torch.long, device=device)
            logits = model(inp_tensor)
            next_logits = logits[0, -1, :] / temperature
            
            if top_k > 0:
                v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < v[-1]] = -float('Inf')
                
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_token)
            
            if next_token == tokenizer.eos_token_id:
                break
                
    return tokenizer.decode(generated)

prompts = [
    "Once upon a time, a little girl named Lily",
    "One day, a tiny puppy saw a big ball in the garden.",
    "Tom wanted to build a high tower with his wooden blocks."
]

print("=" * 70)
print("  ENGRAMA 30M - AUTOREGRESSIVE STORY GENERATION SAMPLES")
print("=" * 70)

for i, prompt in enumerate(prompts, 1):
    story = generate_story(prompt, max_new_tokens=80, temperature=0.7)
    print(f"\n--- Sample {i} ---")
    print(story)
    print("-" * 50)


In [ ]:
# Step 10: Save Trained Checkpoint & Configuration
os.makedirs("checkpoints", exist_ok=True)

save_path = "checkpoints/engrama_30m_tinystories.pt"
config_path = "checkpoints/config.json"

raw_model = train_model.module if isinstance(train_model, nn.DataParallel) else train_model

torch.save({
    "model_state_dict": raw_model.state_dict(),
    "config": asdict(config),
    "vocab_size": config.vocab_size,
    "d_model": config.d_model,
}, save_path)

with open(config_path, "w") as f:
    json.dump(asdict(config), f, indent=2)

print(f"✅ Trained Engrama 30M checkpoint saved successfully to: {save_path}")
print(f"✅ Configuration JSON saved to: {config_path}")
